In [1]:
import os
import rasterio
from pathlib import Path
from tqdm import tqdm

from beak.experimental.utilities.raster_processing import unify_raster_grids
from beak.experimental.utilities.io import save_raster, create_file_list, check_path, data_folder


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

In [6]:
BASE_PATH = data_folder()
EPSG_CODE = 102008
RESOLUTION = 100

BASE_EXTENT = "tusk_great_basin_a2"
BASE_RASTER = BASE_PATH / "processed" / str(f"regional_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "base_raster" / "template_raster_geodawn_area2.tif"
BASE_EVIDENCE = "geophysics"
CMA_EXTENT = "regional"

# Export
# If some data sets are already **LOG** scaled they need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "raw" / BASE_EVIDENCE / "regional_scale" / "tusk_great_basin_nv"
PATH_EXPORT = BASE_PATH / "processed" / str(f"{CMA_EXTENT}_{BASE_EXTENT}_{EPSG_CODE}_{RESOLUTION}") / "unified" / BASE_EVIDENCE

CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\raw\geophysics\regional_scale\tusk_great_basin_nv
Export folder: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\processed\regional_tusk_great_basin_a2_102008_100\unified\geophysics


Select subfolders to be scaled.

In [7]:
for folder in os.listdir(PATH_INPUT):
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


emri


In [8]:
SELECTION = ["emri"]

input_folders = [PATH_INPUT / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

Selected folders:
S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\raw\geophysics\regional_scale\tusk_great_basin_nv\emri


**Select files**

In [9]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Show results
print(f"Found {len(file_list)} files.")


Found 13 files.


**Run unification**

In [10]:
for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    unified_raster, unified_meta = unify_raster_grids(BASE_RASTER, [file], resampling_mode="auto", same_extent=True, same_shape=True)[0]

    output_folder = OUT_FOLDER / str(folder_relative)
    out_path = output_folder / file.name
    check_path(Path(os.path.dirname(out_path)))
    save_raster(out_path, array=unified_raster, dtype="float32", metadata=unified_meta)


100%|██████████| 13/13 [00:09<00:00,  1.33it/s]
